In [27]:
import pandas as pd
import numpy as np

np.random.seed(42)

dataset_path = '../assets/processed/'
X_train, X_dev, X_test, = pd.read_csv(f'{dataset_path}X_train.csv'), pd.read_csv(f'{dataset_path}X_dev.csv'), pd.read_csv(f'{dataset_path}X_test.csv')
Y_train, Y_dev, Y_test, = pd.read_csv(f'{dataset_path}Y_train.csv'), pd.read_csv(f'{dataset_path}Y_dev.csv'), pd.read_csv(f'{dataset_path}Y_test.csv')

X_train, X_dev, X_test = X_train.to_numpy(dtype='float32').T, X_dev.to_numpy(dtype='float32').T, X_test.to_numpy(dtype='float32').T # (n_features, n_samples)
Y_train, Y_dev, Y_test = Y_train.to_numpy(dtype='float32').T, Y_dev.to_numpy(dtype='float32').T, Y_test.to_numpy(dtype='float32').T # (1, n_samples)

### Parameters initialization

In [36]:
learning_rate = 0.001
batch_size = 128 # Minibatch size
epochs = 100
patience = 10 # early stopping: epochs without dev-loss improvement before stopping
units = [X_train.shape[0], 32, 16, 1] # input, hidden, output layers
activation = 'relu'
beta_momentum = 0.9 # Adam 1st moment decay (beta1)
beta_rmsprop = 0.999 # Adam 2nd moment decay (beta2)
epsilon = 1e-8 # small value to avoid division by zero
weight_decay = 1e-2 # AdamW decoupled weight decay

parameters = {}
for layer in range(1, len(units)):
    # W: (layer units, layer-1 units)
    parameters[f'W{layer}'] = np.random.randn(units[layer], units[layer - 1]) * np.sqrt(2 / units[layer - 1]) # He initialization
    parameters[f'b{layer}'] = np.zeros((units[layer], 1))

adam_parameters = {}
for layer in range(1, len(units)):
    adam_parameters[f'VdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Vdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])
    adam_parameters[f'SdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Sdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])

In [29]:
def activation_func(activaton, Z):
    if activaton == 'relu':
        return np.maximum(0, Z)
    if activaton == 'sigmoid':
        return 1 / (1 + np.exp(-Z))

In [30]:
def derivative_activation_func(activaton, Z):
    if activaton == 'relu':
        return np.where(Z > 0, 1, 0) # derivative of ReLU (1 if Z > 0 else 0)

In [31]:
def loss_func(Y_hat, Y):
    m = Y.shape[1]
    cross_entropy_loss = -1/m * np.sum(Y * np.log(Y_hat + 1e-8) + (1 - Y) * np.log(1 - Y_hat + 1e-8)) # epsilon to avoid log(0)
    return cross_entropy_loss

In [32]:
def forward_propagation(X, L, activation, parameters):
    cache = {}
    A = X # A0 (input layer)

    for layer in range(1, L + 1):
        W = parameters[f'W{layer}']
        b = parameters[f'b{layer}']
        
        Z = np.dot(W, A) + b # Z = W * A + b
        A = activation_func(activation, Z) if layer < L else activation_func('sigmoid', Z) # sigmoid for the output layer
        cache[f'Z{layer}'], cache[f'A{layer}'] = Z, A

    return A, cache

In [33]:
def backward_propagation(X, dZ, m, L, activation, parameters, cache):
    grads = {}

    for layer in reversed(range(1, L + 1)):
        A_prev = cache[f'A{layer - 1}'] if layer > 1 else X
        W = parameters[f'W{layer}']
        
        grads[f'dW{layer}'] = (1 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{layer}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

        if layer > 1:
            dA_prev = np.dot(W.T, dZ)
            dZ = dA_prev * derivative_activation_func(activation, cache[f'Z{layer - 1}'])

    return grads

In [34]:
def evaluate(X, Y, L, activation, parameters):
    Y_hat, _ = forward_propagation(X, L, activation, parameters)
    loss = loss_func(Y_hat, Y)
    accuracy = ((Y_hat > 0.5) == Y).mean() 
    return loss, accuracy

### Model training

In [37]:
t = 0 # Adam optimization time step
L = len(units) - 1
m_train = X_train.shape[1]

best_dev_loss = np.inf
best_parameters = None
epochs_no_improve = 0

for epoch in range(epochs):
    # shuffle the training set each epoch, then split into minibatches
    perm = np.random.permutation(m_train)
    X_shuf, Y_shuf = X_train[:, perm], Y_train[:, perm]
    batches = [(X_shuf[:, i:i + batch_size], Y_shuf[:, i:i + batch_size])
               for i in range(0, m_train, batch_size)]

    for X_batch, Y_batch in batches:
        m = X_batch.shape[1] # samples in this batch

        # forward prop
        Y_hat, cache = forward_propagation(X_batch, L, activation, parameters)

        # loss (pure cross-entropy; weight decay is decoupled, AdamW style)
        loss = loss_func(Y_hat, Y_batch)

        # back prop
        dZ = Y_hat - Y_batch # sigmoid + BCE chain derivative (last layer)
        grads = backward_propagation(X_batch, dZ, m, L, activation, parameters, cache)

        # AdamW parameter update
        t += 1
        for layer in range(1, L + 1):
            dW = grads[f'dW{layer}']
            db = grads[f'db{layer}']

            # 1st (momentum) and 2nd (RMSProp) moment estimates
            adam_parameters[f'VdW{layer}'] = beta_momentum * adam_parameters[f'VdW{layer}'] + (1 - beta_momentum) * dW
            adam_parameters[f'Vdb{layer}'] = beta_momentum * adam_parameters[f'Vdb{layer}'] + (1 - beta_momentum) * db
            adam_parameters[f'SdW{layer}'] = beta_rmsprop * adam_parameters[f'SdW{layer}'] + (1 - beta_rmsprop) * (dW ** 2)
            adam_parameters[f'Sdb{layer}'] = beta_rmsprop * adam_parameters[f'Sdb{layer}'] + (1 - beta_rmsprop) * (db ** 2)

            # bias correction
            VdW_corr = adam_parameters[f'VdW{layer}'] / (1 - beta_momentum ** t)
            Vdb_corr = adam_parameters[f'Vdb{layer}'] / (1 - beta_momentum ** t)
            SdW_corr = adam_parameters[f'SdW{layer}'] / (1 - beta_rmsprop ** t)
            Sdb_corr = adam_parameters[f'Sdb{layer}'] / (1 - beta_rmsprop ** t)

            dW_adam = VdW_corr / (np.sqrt(SdW_corr) + epsilon) # Adam update for weights
            db_adam = Vdb_corr / (np.sqrt(Sdb_corr) + epsilon) # Adam update for bias

            parameters[f'W{layer}'] -= learning_rate * (dW_adam + weight_decay * parameters[f'W{layer}'])
            parameters[f'b{layer}'] -= learning_rate * db_adam

    # early stopping: evaluate dev loss every epoch, keep the best parameters
    dv_loss, dv_acc = evaluate(X_dev, Y_dev, L, activation, parameters)
    if dv_loss < best_dev_loss:
        best_dev_loss = dv_loss
        best_parameters = {k: v.copy() for k, v in parameters.items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 20 == 0 or epochs_no_improve >= patience:
        tr_loss, tr_acc = evaluate(X_train, Y_train, L, activation, parameters)
        print(f'ep {epoch:3d}  '
                f'| train: loss {tr_loss:.4f} acc {tr_acc:.4f} | '
                f'| dev: loss {dv_loss:.4f} acc {dv_acc:.4f} |')

    if epochs_no_improve >= patience:
        print(f'early stopping at epoch {epoch} (best dev loss {best_dev_loss:.4f})')
        break

# restore the best-generalizing weights
parameters = best_parameters

ep   0  | train: loss 0.6521 acc 0.6176 | | dev: loss 0.6579 acc 0.6068 |
ep  20  | train: loss 0.6437 acc 0.6246 | | dev: loss 0.6548 acc 0.6112 |
ep  25  | train: loss 0.6431 acc 0.6258 | | dev: loss 0.6551 acc 0.6112 |
early stopping at epoch 25 (best dev loss 0.6545)
